# Duplicates and Outliers

Explore duplicate rows, redundant `*_name` columns, and extreme income / loan amounts in the NY HMDA 2015 data.

This notebook is **read-only analysis**. Cleaning and writing CSVs live in `scripts/` — run `python main.py` from the project root to update data.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = os.path.join('..', 'data', 'ny_hmda_2015.csv')

df = pd.read_csv(DATA_PATH, low_memory=False)
print(df.shape)
df.head(3)

(439654, 78)


,action_taken,action_taken_name,agency_code,agency_abbr,agency_name,applicant_ethnicity,applicant_ethnicity_name,applicant_income_000s,applicant_race_1,applicant_race_2,...,state_abbr,state_name,hud_median_family_income,loan_amount_000s,number_of_1_to_4_family_units,number_of_owner_occupied_units,minority_population,population,rate_spread,tract_to_msamd_income
0,1,Loan originated,9,CFPB,Consumer Financial Protection Bureau,2,Not Hispanic or Latino,97.0,5,NaN,...,NY,New York,109000.0,187,363.0,1817.0,21.139999,5870.0,NaN,109.459999
1,1,Loan originated,9,CFPB,Consumer Financial Protection Bureau,2,Not Hispanic or Latino,200.0,5,NaN,...,NY,New York,71300.0,460,53.0,256.0,45.959999,3512.0,NaN,160.600006
2,1,Loan originated,7,HUD,Department of Housing and Urban Development,2,Not Hispanic or Latino,NaN,3,NaN,...,NY,New York,71300.0,296,2745.0,2586.0,38.990002,8357.0,NaN,134.820007


## Duplicate Rows

In [2]:
n_exact = df.duplicated().sum()
print(f'Exact duplicate rows: {n_exact}')
print('Shape:', df.shape)

Exact duplicate rows: 0
Shape: (439654, 78)


## Redundant Code / Name Columns

Many HMDA fields appear twice (numeric code + human-readable name). Scripts keep codes for modeling and drop the matching `*_name` columns — here we only list them.

In [3]:
name_cols = [c for c in df.columns if '_name' in c]
print(f'Redundant name columns ({len(name_cols)}):')
print(name_cols)
print(f'Shape if dropped: ({df.shape[0]}, {df.shape[1] - len(name_cols)})')

Redundant name columns (31):
['action_taken_name', 'agency_name', 'applicant_ethnicity_name', 'applicant_race_name_1', 'applicant_race_name_2', 'applicant_race_name_3', 'applicant_race_name_4', 'applicant_race_name_5', 'applicant_sex_name', 'co_applicant_ethnicity_name', 'co_applicant_race_name_1', 'co_applicant_race_name_2', 'co_applicant_race_name_3', 'co_applicant_race_name_4', 'co_applicant_race_name_5', 'co_applicant_sex_name', 'county_name', 'denial_reason_name_1', 'denial_reason_name_2', 'denial_reason_name_3', 'edit_status_name', 'hoepa_status_name', 'lien_status_name', 'loan_purpose_name', 'loan_type_name', 'msamd_name', 'owner_occupancy_name', 'preapproval_name', 'property_type_name', 'purchaser_type_name', 'state_name']
Shape if dropped: (439654, 47)


## Outliers

Focus on `applicant_income_000s` and `loan_amount_000s`. Census / MSA fields are left alone — extreme values there are usually valid neighborhood traits, not errors.

In [4]:
outlier_cols = ['applicant_income_000s', 'loan_amount_000s']
df[outlier_cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T

,count,mean,std,min,1%,25%,50%,75%,99%,max
applicant_income_000s,378651.0,140.145794,268.471316,1.0,17.0,58.0,90.0,142.0,1000.0,9999.0
loan_amount_000s,439654.0,333.324287,1173.204181,1.0,4.0,102.0,208.0,366.0,2280.0,99999.0


### IQR outlier counts

In [5]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in outlier_cols:
    s = df[col].dropna()
    lo, hi = iqr_bounds(s)
    n_out = ((s < lo) | (s > hi)).sum()
    print(f'{col}: {n_out} outside [{lo:.1f}, {hi:.1f}] ({100 * n_out / len(s):.1f}%)')

applicant_income_000s: 30806 outside [-68.0, 268.0] (8.1%)
loan_amount_000s: 23898 outside [-294.0, 762.0] (5.4%)


### What 1st / 99th percentile capping would do

Scripts winsorize these columns to keep sample size and reduce the pull of extreme incomes / loan amounts (including top-coded values like 9999 / 99999). Below we only report how many values would be capped — `df` is not modified.

In [6]:
caps = {}
for col in outlier_cols:
    lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
    caps[col] = (lo, hi)
    n_would_cap = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f'{col}: would cap {n_would_cap} values to [{lo:.1f}, {hi:.1f}]')
    preview = df[col].clip(lower=lo, upper=hi)
    print(f'  preview mean {preview.mean():.2f} (raw mean {df[col].mean():.2f}), preview max {preview.max():.1f}')

print('\nRaw distributions:')
df[outlier_cols].describe().T

applicant_income_000s: would cap 7485 values to [17.0, 1000.0]
  preview mean 129.81 (raw mean 140.15), preview max 1000.0
loan_amount_000s: would cap 8349 values to [4.0, 2280.0]
  preview mean 292.88 (raw mean 333.32), preview max 2280.0

Raw distributions:


,count,mean,std,min,25%,50%,75%,max
applicant_income_000s,378651.0,140.145794,268.471316,1.0,58.0,90.0,142.0,9999.0
loan_amount_000s,439654.0,333.324287,1173.204181,1.0,102.0,208.0,366.0,99999.0


## Data updates

No CSV is written from this notebook. To apply duplicate / name-column / outlier cleaning, run from the project root:

```bash
python main.py
```

That runs `scripts/duplicate_handler.py`, `scripts/outlier_handler.py`, and the rest of the prep pipeline.

In [7]:
print('Explore only — no file written.')
print(f'Current in-memory shape: {df.shape[0]} rows × {df.shape[1]} cols')
print('Cleaned outputs are produced by scripts via: python main.py')

Explore only — no file written.
Current in-memory shape: 439654 rows × 78 cols
Cleaned outputs are produced by scripts via: python main.py
